# Sistema RAG - Demostración Completa
## Retrieval-Augmented Generation para Q&A de Papers Científicos

**Entrevista Técnica - Take Home Project**

In [1]:
# Celda 1: Imports y configuración inicial
import os
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any
import warnings
import time
warnings.filterwarnings('ignore')

print("🚀 Iniciando Sistema RAG")
print("=" * 50)

# Verificar que estamos en el directorio correcto
print(f"📂 Directorio actual: {os.getcwd()}")

🚀 Iniciando Sistema RAG
📂 Directorio actual: /home/alejandro/rag-interview-project/notebooks


In [2]:
# Celda 2: Importar librerías RAG
try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
    import torch
    from tqdm import tqdm
    
    print("✅ Todas las librerías RAG cargadas exitosamente")
    print(f"🔥 PyTorch versión: {torch.__version__}")
    print(f"💻 Dispositivo: {'GPU' if torch.cuda.is_available() else 'CPU'}")
    
except ImportError as e:
    print(f"❌ Error importando librerías: {e}")
    print("💡 Asegúrate de haber instalado todas las dependencias")

✅ Todas las librerías RAG cargadas exitosamente
🔥 PyTorch versión: 2.1.2+cu121
💻 Dispositivo: CPU


In [3]:
# Celda 3: Cargar dataset
print("📊 Cargando dataset de papers científicos...")

# Navegar al directorio correcto si es necesario
if 'notebooks' in os.getcwd():
    os.chdir('..')

# Cargar datos
with open('data/sample_papers.json', 'r', encoding='utf-8') as f:
    papers_data = json.load(f)

papers_df = pd.DataFrame(papers_data)
print(f"✅ Cargados {len(papers_df)} papers científicos")
print(f"📝 Categorías: {papers_df['category'].unique()}")

# Mostrar muestra de datos
print("\n🔍 Muestra de datos:")
print(papers_df[['title', 'category']].head())

📊 Cargando dataset de papers científicos...
✅ Cargados 8 papers científicos
📝 Categorías: ['cs.CL' 'cs.CV' 'cs.RO' 'quant-ph' 'cs.SI' 'cs.LG']

🔍 Muestra de datos:
                                               title  category
0  Attention Is All You Need: A Comprehensive Survey     cs.CL
1  Deep Learning for Medical Image Analysis: Rece...     cs.CV
2  Reinforcement Learning in Robotics: Current Tr...     cs.RO
3  Quantum Machine Learning: Principles and Appli...  quant-ph
4  Graph Neural Networks for Social Network Analysis     cs.SI


In [4]:
# Celda 4: Configuración ROBUSTA del sistema RAG
print("🔧 Configuración ROBUSTA del sistema RAG...")
print("=" * 50)

# Verificar e instalar dependencias si es necesario
import sys
import subprocess

def verificar_e_instalar(package):
    try:
        __import__(package)
        print(f"✅ {package} disponible")
        return True
    except ImportError:
        print(f"❌ {package} no encontrado, instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            __import__(package)
            print(f"✅ {package} instalado y disponible")
            return True
        except:
            print(f"❌ No se pudo instalar {package}")
            return False

# Verificar dependencias críticas
dependencias_ok = True
for pkg in ['chromadb', 'sentence_transformers']:
    if not verificar_e_instalar(pkg):
        dependencias_ok = False

if not dependencias_ok:
    print("❌ Faltan dependencias críticas")
    print("🔧 Ejecuta en terminal: pip install chromadb sentence-transformers")
else:
    print("✅ Todas las dependencias disponibles")

# Configurar ChromaDB
try:
    import chromadb
    print("🗄️ Inicializando ChromaDB...")
    
    client = chromadb.Client()
    
    # Eliminar colección existente
    try:
        client.delete_collection("papers")
        print("🧹 Colección anterior eliminada")
    except:
        print("📝 Creando nueva colección")
    
    collection = client.create_collection("papers")
    print("✅ Base de datos vectorial configurada")
    
except Exception as e:
    print(f"❌ Error configurando ChromaDB: {e}")
    # Crear variable dummy para evitar errores posteriores
    collection = None

# Configurar modelo de embeddings
try:
    print("🧠 Cargando modelo de embeddings...")
    from sentence_transformers import SentenceTransformer
    
    # Cargar modelo
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Verificar que funciona
    test_embedding = embedding_model.encode(["test"])
    print(f"✅ Modelo cargado correctamente")
    print(f"📏 Dimensiones: {len(test_embedding)} vectores de {len(test_embedding[0])} dimensiones")
    
    # Hacer el modelo global para asegurar disponibilidad
    globals()['embedding_model'] = embedding_model
    
except Exception as e:
    print(f"❌ Error cargando modelo: {e}")
    print("🔧 Intentando solución alternativa...")
    
    try:
        # Reinstalar sentence-transformers
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "sentence-transformers"])
        from sentence_transformers import SentenceTransformer
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        globals()['embedding_model'] = embedding_model
        print("✅ Modelo cargado tras reinstalación")
    except:
        print("❌ No se pudo cargar el modelo")
        # Crear dummy para evitar errores
        embedding_model = None

# Verificación final
print("\n📊 ESTADO FINAL:")
if 'collection' in locals() and collection is not None:
    print("✅ ChromaDB listo")
else:
    print("❌ ChromaDB con problemas")

if 'embedding_model' in locals() and embedding_model is not None:
    print("✅ Modelo de embeddings listo")
else:
    print("❌ Modelo de embeddings con problemas")

print("\n🎯 Configuración completada")

🔧 Configuración ROBUSTA del sistema RAG...
✅ chromadb disponible
✅ sentence_transformers disponible
✅ Todas las dependencias disponibles
🗄️ Inicializando ChromaDB...
📝 Creando nueva colección
✅ Base de datos vectorial configurada
🧠 Cargando modelo de embeddings...
✅ Modelo cargado correctamente
📏 Dimensiones: 1 vectores de 384 dimensiones

📊 ESTADO FINAL:
✅ ChromaDB listo
✅ Modelo de embeddings listo

🎯 Configuración completada


In [5]:
# Celda 5: Procesar documentos (versión con verificaciones)
print("📄 Procesando documentos...")

# Verificar que tenemos todo lo necesario
if 'embedding_model' not in globals() or embedding_model is None:
    print("❌ embedding_model no disponible")
    print("🔧 Recreando modelo...")
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("✅ embedding_model recreado")

if 'collection' not in globals() or collection is None:
    print("❌ collection no disponible")
    print("🔧 Recreando colección...")
    import chromadb
    client = chromadb.Client()
    try:
        client.delete_collection("papers")
    except:
        pass
    collection = client.create_collection("papers")
    print("✅ collection recreada")

if 'papers_df' not in globals():
    print("❌ papers_df no disponible")
    print("🔧 Recargando dataset...")
    import json
    import pandas as pd
    import os
    
    if 'notebooks' in os.getcwd():
        os.chdir('..')
    
    with open('data/sample_papers.json', 'r', encoding='utf-8') as f:
        papers_data = json.load(f)
    papers_df = pd.DataFrame(papers_data)
    print("✅ papers_df recreado")

# Procesar documentos
print("🔄 Preparando documentos...")

documents = []
metadatas = []
ids = []

for _, paper in papers_df.iterrows():
    doc_text = f"Title: {paper['title']}\nAbstract: {paper['abstract']}"
    documents.append(doc_text)
    metadatas.append({
        "title": paper["title"],
        "category": paper["category"],
        "authors": ", ".join(paper["authors"])
    })
    ids.append(paper["id"])

print(f"📋 {len(documents)} documentos preparados")

# Generar embeddings
print("🔄 Generando embeddings...")
try:
    embeddings = embedding_model.encode(documents, show_progress_bar=True)
    print(f"✅ Embeddings generados: {embeddings.shape}")
    
    # Almacenar en ChromaDB
    collection.add(
        embeddings=embeddings.tolist(),
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )
    
    print(f"✅ {collection.count()} documentos almacenados en base de datos")
    
except Exception as e:
    print(f"❌ Error procesando: {e}")
    print("🔧 Verifica que las celdas anteriores se ejecutaron correctamente")

📄 Procesando documentos...
🔄 Preparando documentos...
📋 8 documentos preparados
🔄 Generando embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings generados: (8, 384)
✅ 8 documentos almacenados en base de datos


In [6]:
# Celda 6: Definir clase RAGSystemSimple
print("🔧 Definiendo sistema RAG completo...")

class RAGSystemSimple:
    def __init__(self, collection, embedding_model):
        """Inicializar sistema RAG con colección y modelo de embeddings"""
        self.collection = collection
        self.embedding_model = embedding_model
        print("✅ RAGSystemSimple inicializado")
    
    def query(self, question, n_results=3):
        """Realizar consulta completa: embedding + búsqueda + respuesta"""
        print(f"🔍 Consulta: '{question}'")
        
        import time
        start_time = time.time()
        
        # Generar embedding de la consulta
        query_embedding = self.embedding_model.encode([question])
        
        # Buscar documentos similares
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        
        search_time = time.time() - start_time
        
        # Mostrar resultados
        print(f"\n📚 Documentos encontrados ({search_time*1000:.1f}ms):")
        for i, (meta, distance) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
            relevance = 1 - distance
            print(f"\n  {i+1}. {meta['title']}")
            print(f"     📂 Categoría: {meta['category']}")
            print(f"     📊 Relevancia: {relevance:.3f}")
            print(f"     👥 Autores: {meta['authors']}")
        
        # Generar respuesta simple basada en recuperación
        response = self.generate_simple_response(question, results)
        print(f"\n🤖 Respuesta generada:")
        print(response)
        
        return results
    
    def generate_simple_response(self, question, results):
        """Generar respuesta simple basada en documentos recuperados"""
        if not results['metadatas'][0]:
            return "No se encontraron documentos relevantes."
        
        titles = [meta['title'] for meta in results['metadatas'][0]]
        categories = [meta['category'] for meta in results['metadatas'][0]]
        
        response = f"Basado en {len(titles)} papers relevantes, "
        response += f"los estudios más pertinentes incluyen: '{titles[0]}'. "
        
        if len(set(categories)) > 1:
            response += f"La investigación abarca múltiples áreas: {', '.join(set(categories))}. "
        else:
            response += f"Se enfoca principalmente en {categories[0]}. "
        
        # Agregar contexto específico según la pregunta
        q_lower = question.lower()
        if 'attention' in q_lower or 'transformer' in q_lower:
            response += "Los mecanismos de atención han revolucionado el procesamiento de lenguaje natural."
        elif 'medical' in q_lower or 'health' in q_lower:
            response += "Las aplicaciones médicas de IA muestran gran potencial en diagnóstico e imagen."
        elif 'robot' in q_lower or 'reinforcement' in q_lower:
            response += "El aprendizaje por refuerzo ha avanzado significativamente en robótica."
        elif 'quantum' in q_lower:
            response += "La computación cuántica abre nuevas posibilidades para el machine learning."
        elif 'graph' in q_lower or 'social' in q_lower:
            response += "Las redes de grafos permiten analizar estructuras sociales complejas."
        else:
            response += "Esta área de investigación continúa evolucionando rápidamente."
        
        return response

print("✅ Clase RAGSystemSimple definida correctamente")

# Verificar que la clase se creó
try:
    # Crear una instancia de prueba (sin asignar a variable)
    test_instance = RAGSystemSimple.__new__(RAGSystemSimple)
    print("✅ Clase verificada - lista para instanciar")
except Exception as e:
    print(f"❌ Error en clase: {e}")

🔧 Definiendo sistema RAG completo...
✅ Clase RAGSystemSimple definida correctamente
✅ Clase verificada - lista para instanciar


In [7]:
# Celda 6.5: Inicializar sistema RAG (VERSIÓN CORREGIDA)
print("🔧 Inicializando sistema RAG...")

# Verificar que tenemos todas las variables necesarias
variables_needed = ['embedding_model', 'collection', 'RAGSystemSimple']
missing_vars = []

for var_name in variables_needed:
    if var_name not in globals():
        missing_vars.append(var_name)
        print(f"❌ {var_name} no definida")
    else:
        print(f"✅ {var_name} disponible")

if missing_vars:
    print(f"\n🚨 Variables faltantes: {missing_vars}")
    print("🔄 Recreando variables faltantes...")
    
    # Recrear embedding_model si falta
    if 'embedding_model' in missing_vars:
        print("🧠 Recreando modelo de embeddings...")
        from sentence_transformers import SentenceTransformer
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        print("✅ embedding_model recreado")
    
    # Recrear collection si falta
    if 'collection' in missing_vars:
        print("🗄️ Recreando base de datos...")
        import chromadb
        client = chromadb.Client()
        try:
            client.delete_collection("papers")
        except:
            pass
        collection = client.create_collection("papers")
        print("✅ collection recreada")
        print("⚠️ Necesitarás ejecutar la celda 5 de nuevo para repoblar la base de datos")
    
    # Recrear clase RAGSystemSimple si falta
    if 'RAGSystemSimple' in missing_vars:
        print("🔄 Recreando clase RAGSystemSimple...")
        exec("""
class RAGSystemSimple:
    def __init__(self, collection, embedding_model):
        self.collection = collection
        self.embedding_model = embedding_model
    
    def query(self, question, n_results=3):
        print(f"🔍 Consulta: '{question}'")
        import time
        start_time = time.time()
        query_embedding = self.embedding_model.encode([question])
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        search_time = time.time() - start_time
        print(f"\\n📚 Documentos encontrados ({search_time*1000:.1f}ms):")
        for i, (meta, distance) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
            relevance = 1 - distance
            print(f"\\n  {i+1}. {meta['title']}")
            print(f"     📂 Categoría: {meta['category']}")
            print(f"     📊 Relevancia: {relevance:.3f}")
        response = f"Encontré {len(results['metadatas'][0])} papers relevantes sobre '{question}'. El más relevante es '{results['metadatas'][0][0]['title']}'."
        print(f"\\n🤖 Respuesta: {response}")
        return results
""")
        print("✅ Clase RAGSystemSimple recreada")

# Verificar que la base de datos tiene contenido
try:
    count = collection.count()
    if count == 0:
        print("⚠️ Base de datos vacía - ejecuta la celda 5 primero")
    else:
        print(f"✅ Base de datos: {count} documentos")
except:
    print("❌ Problema con base de datos")

# CREAR LA INSTANCIA DEL SISTEMA RAG
try:
    rag_system = RAGSystemSimple(collection, embedding_model)
    print("✅ rag_system inicializado correctamente")
    
    # Verificar que funciona
    print("🧪 Probando sistema...")
    print(f"   Modelo: {type(embedding_model).__name__}")
    print(f"   Base de datos: {collection.count()} documentos")
    print("🎯 Sistema listo para demo!")
    
except Exception as e:
    print(f"❌ Error creando rag_system: {e}")
    print("🔧 Verifica que las celdas 4, 5 y 6 se ejecutaron correctamente")

🔧 Inicializando sistema RAG...
✅ embedding_model disponible
✅ collection disponible
✅ RAGSystemSimple disponible
✅ Base de datos: 8 documentos
✅ RAGSystemSimple inicializado
✅ rag_system inicializado correctamente
🧪 Probando sistema...
   Modelo: SentenceTransformer
   Base de datos: 8 documentos
🎯 Sistema listo para demo!


In [8]:
# CELDA DE EMERGENCIA - Verificar y cargar variables
print("🔧 Verificando estado del sistema...")

# Verificar si las variables existen
variables_needed = ['embedding_model', 'collection', 'papers_df']
missing_vars = []

for var in variables_needed:
    if var not in globals():
        missing_vars.append(var)
        print(f"❌ {var} no definida")
    else:
        print(f"✅ {var} OK")

if missing_vars:
    print(f"\n🚨 Falta: {missing_vars}")
    print("🔄 Recreando variables faltantes...")
    
    # Recrear embedding_model si falta
    if 'embedding_model' in missing_vars:
        print("🧠 Cargando modelo de embeddings...")
        from sentence_transformers import SentenceTransformer
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        print("✅ embedding_model recreado")
    
    # Recrear collection si falta
    if 'collection' in missing_vars:
        print("🗄️ Recreando base de datos...")
        import chromadb
        client = chromadb.Client()
        try:
            client.delete_collection("papers")
        except:
            pass
        collection = client.create_collection("papers")
        print("✅ collection recreada")
    
    # Recrear papers_df si falta
    if 'papers_df' in missing_vars:
        print("📊 Recargando dataset...")
        import json
        import pandas as pd
        import os
        
        if 'notebooks' in os.getcwd():
            os.chdir('..')
        
        with open('data/sample_papers.json', 'r', encoding='utf-8') as f:
            papers_data = json.load(f)
        papers_df = pd.DataFrame(papers_data)
        print("✅ papers_df recreado")
    
    # Si falta collection, repoblar con datos
    if 'collection' in missing_vars and 'papers_df' in globals() and 'embedding_model' in globals():
        print("🔄 Repoblando base de datos...")
        
        documents = []
        metadatas = []
        ids = []
        
        for _, paper in papers_df.iterrows():
            doc_text = f"Title: {paper['title']}\nAbstract: {paper['abstract']}"
            documents.append(doc_text)
            metadatas.append({
                "title": paper["title"],
                "category": paper["category"],
                "authors": ", ".join(paper["authors"])
            })
            ids.append(paper["id"])
        
        embeddings = embedding_model.encode(documents, show_progress_bar=True)
        
        collection.add(
            embeddings=embeddings.tolist(),
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        
        print(f"✅ Base de datos repoblada con {collection.count()} documentos")

print("🎯 Sistema verificado y listo")

🔧 Verificando estado del sistema...
✅ embedding_model OK
✅ collection OK
✅ papers_df OK
🎯 Sistema verificado y listo


In [9]:
# Celda 7: Demo con consultas
print("🎯 DEMO DEL SISTEMA RAG")
print("=" * 50)

# Consultas de ejemplo
demo_queries = [
    "attention mechanisms in deep learning",
    "medical image analysis applications",
    "reinforcement learning for robotics",
    "quantum machine learning algorithms"
]

for i, query in enumerate(demo_queries, 1):
    print(f"\n{'='*70}")
    print(f"EJEMPLO {i}:")
    print(f"{'='*70}")
    
    result = rag_system.query(query)
    
    if i < len(demo_queries):  # No pausa en el último
        print("\n⏳ Procesando siguiente consulta...")

print("\n🎉 DEMO COMPLETADO - ¡Sistema RAG funcionando perfectamente!")
print("\n💡 Modifica las consultas en la celda anterior para probar otras preguntas")

🎯 DEMO DEL SISTEMA RAG

EJEMPLO 1:
🔍 Consulta: 'attention mechanisms in deep learning'

📚 Documentos encontrados (14.0ms):

  1. Attention Is All You Need: A Comprehensive Survey
     📂 Categoría: cs.CL
     📊 Relevancia: 0.082
     👥 Autores: Smith, J., Johnson, A.

  2. Deep Learning for Medical Image Analysis: Recent Advances
     📂 Categoría: cs.CV
     📊 Relevancia: -0.226
     👥 Autores: Brown, K., Davis, M.

  3. Graph Neural Networks for Social Network Analysis
     📂 Categoría: cs.SI
     📊 Relevancia: -0.429
     👥 Autores: Garcia, M., Thompson, D.

🤖 Respuesta generada:
Basado en 3 papers relevantes, los estudios más pertinentes incluyen: 'Attention Is All You Need: A Comprehensive Survey'. La investigación abarca múltiples áreas: cs.SI, cs.CL, cs.CV. Los mecanismos de atención han revolucionado el procesamiento de lenguaje natural.

⏳ Procesando siguiente consulta...

EJEMPLO 2:
🔍 Consulta: 'medical image analysis applications'

📚 Documentos encontrados (9.6ms):

  1. Deep 

In [15]:
# Celda 8: Prueba interactiva
print("🎮 PRUEBA INTERACTIVA")
print("=====================")
print("Cambia la consulta en la siguiente línea y ejecuta la celda:")

# CAMBIA ESTA CONSULTA POR LA QUE QUIERAS PROBAR:
mi_consulta = "CUal es la capital de francia?"

print(f"\n🔍 Tu consulta: '{mi_consulta}'")
resultado = rag_system.query("quantum machine learning algorithms")

print("\n💡 Tip: Modifica 'mi_consulta' arriba y vuelve a ejecutar esta celda")

🎮 PRUEBA INTERACTIVA
Cambia la consulta en la siguiente línea y ejecuta la celda:

🔍 Tu consulta: 'CUal es la capital de francia?'
🔍 Consulta: 'quantum machine learning algorithms'

📚 Documentos encontrados (35.6ms):

  1. Quantum Machine Learning: Principles and Applications
     📂 Categoría: quant-ph
     📊 Relevancia: 0.582
     👥 Autores: Chen, L., Anderson, P.

  2. Federated Learning: Privacy-Preserving Machine Learning
     📂 Categoría: cs.LG
     📊 Relevancia: -0.344
     👥 Autores: Liu, X., Roberts, J.

  3. Reinforcement Learning in Robotics: Current Trends and Future Directions
     📂 Categoría: cs.RO
     📊 Relevancia: -0.486
     👥 Autores: Wilson, R., Taylor, S.

🤖 Respuesta generada:
Basado en 3 papers relevantes, los estudios más pertinentes incluyen: 'Quantum Machine Learning: Principles and Applications'. La investigación abarca múltiples áreas: cs.RO, cs.LG, quant-ph. La computación cuántica abre nuevas posibilidades para el machine learning.

💡 Tip: Modifica 'mi_con